In [23]:
from datetime import datetime
import json
import sys
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/notebooks/exploration.ipynb"))) # two dirname to get to aml_service path 
    
with open(os.path.join(BASE_DIR, 'config', 'thresholds.json')) as f:
    THRESHOLDS = json.load(f)
sys.path.append(BASE_DIR)

from graph.builder import TransactionsGraph


In [24]:
csv_path = "/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/IBM/amlWORLD/HI-Small_Trans.csv"

In [25]:
import pandas as pd

In [26]:
df = pd.read_csv(csv_path)

In [28]:
df.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [29]:
print("From Bank NaNs:", df["From Bank"].isna().sum())
print("Account NaNs:", df["Account"].isna().sum())
print("To Bank NaNs:", df["To Bank"].isna().sum())
print("Account.1 NaNs:", df["Account.1"].isna().sum())

From Bank NaNs: 0
Account NaNs: 0
To Bank NaNs: 0
Account.1 NaNs: 0


In [30]:
df["From_Account"] = (
    df["From Bank"].astype("string")
    .str.cat(df["Account"].astype("string"), sep="_")
)
df["To_Account"] = (
    df["To Bank"].astype("string")
    .str.cat(df["Account.1"].astype("string"), sep="_")
)

In [31]:
all_accounts = set(df["From_Account"]).union(set(df["To_Account"]))
print("Total unique accounts:", len(all_accounts))

Total unique accounts: 515088


In [8]:
df.drop(columns=["From Bank", "Account", "To Bank", "Account.1"], inplace=True)

In [9]:
df.head()

,Timestamp,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From_Account,To_Account
0,2022/09/01 00:20,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0,10_8000EBD30,10_8000EBD30
1,2022/09/01 00:20,0.01,US Dollar,0.01,US Dollar,Cheque,0,3208_8000F4580,1_8000F5340
2,2022/09/01 00:00,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0,3209_8000F4670,3209_8000F4670
3,2022/09/01 00:02,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0,12_8000F5030,12_8000F5030
4,2022/09/01 00:06,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0,10_8000F5200,10_8000F5200


In [10]:
print(df["Payment Format"].unique())

['Reinvestment' 'Cheque' 'Credit Card' 'ACH' 'Cash' 'Wire' 'Bitcoin']


In [11]:
## print 10 rows where Payment Format
for payment_format in df["Payment Format"].unique():
    print(payment_format)
    #print(df[df["Payment Format"] == payment_format].head(2))
    mask = df["Payment Format"] == payment_format
    all_same = (
        df.loc[mask, "From_Account"]
        == df.loc[mask, "To_Account"]
    ).all()

    print(f"{payment_format}: {all_same}")
    

Reinvestment
Reinvestment: True
Cheque
Cheque: False
Credit Card
Credit Card: False
ACH
ACH: False
Cash
Cash: False
Wire
Wire: False
Bitcoin
Bitcoin: False


In [12]:
print(df["Payment Currency"].unique())

['US Dollar' 'Bitcoin' 'Euro' 'Australian Dollar' 'Yuan' 'Rupee' 'Yen'
 'Mexican Peso' 'UK Pound' 'Ruble' 'Canadian Dollar' 'Swiss Franc'
 'Brazil Real' 'Saudi Riyal' 'Shekel']


In [13]:
currency_exchange = {
    "US Dollar": 52,
    "Bitcoin": 3314084,
    "Euro": 60,
    "Australian Dollar": 36.5,
    "Yuan": 7.67,
    "Rupee": 0.55,
    "Yen": 0.32,
    "Mexican Peso": 3,
    "UK Pound": 69.75,
    "Ruble": 0.72,
    "Canadian Dollar": 37.19,
    "Swiss Franc": 65.27,
    "Brazil Real": 10.24,
    "Saudi Riyal": 13.84,
    "Shekel": 17.8
}

In [ ]:
df["Amount_Paid_EGP"] = df.apply(
    lambda row: row["Amount Paid"] * currency_exchange[row["Payment Currency"]],
    axis=1
)


In [15]:
## nans in amount EGP
print("Amount Paid (EGP) NaNs:", df["Amount Paid (EGP)"].isna().sum())

Amount Paid (EGP) NaNs: 0


In [16]:
## convert Timestamp to datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
print(df["Timestamp"].dtype)

datetime64[ns]


In [17]:
print(max(df["Timestamp"]))
print(min(df["Timestamp"]))
print(df["Timestamp"].max() - df["Timestamp"].min())

2022-09-18 16:18:00
2022-09-01 00:00:00
17 days 16:18:00


In [18]:
print(len(df))

5078345


In [19]:
graph = TransactionsGraph()

for index, row in df.iterrows():
    if index % 10000 == 0:
        print(f"Processing transaction {index} / {len(df)}")
    graph.add_transaction(from_account=row["From_Account"], to_account=row["To_Account"], amount=row["Amount Paid (EGP)"], timestamp=row["Timestamp"])

Processing transaction 0 / 5078345
Processing transaction 10000 / 5078345
Processing transaction 20000 / 5078345
Processing transaction 30000 / 5078345
Processing transaction 40000 / 5078345
Processing transaction 50000 / 5078345
Processing transaction 60000 / 5078345
Processing transaction 70000 / 5078345
Processing transaction 80000 / 5078345
Processing transaction 90000 / 5078345
Processing transaction 100000 / 5078345
Processing transaction 110000 / 5078345
Processing transaction 120000 / 5078345
Processing transaction 130000 / 5078345
Processing transaction 140000 / 5078345
Processing transaction 150000 / 5078345
Processing transaction 160000 / 5078345
Processing transaction 170000 / 5078345
Processing transaction 180000 / 5078345
Processing transaction 190000 / 5078345
Processing transaction 200000 / 5078345
Processing transaction 210000 / 5078345
Processing transaction 220000 / 5078345
Processing transaction 230000 / 5078345
Processing transaction 240000 / 5078345
Processing tra

In [22]:
x = 0
i = 0
total = len(graph.graph.nodes)
for account in graph.graph.nodes:
    if i % 10 == 0:
        print(f"Processing account {i} / {total}")
    if i == 10000:
        break
    i += 1
    x = max(x, (graph.fast_get_money_cycled(account)))
print(x)

Processing account 0 / 515088
Processing account 10 / 515088
Processing account 20 / 515088
Processing account 30 / 515088
Processing account 40 / 515088
Processing account 50 / 515088
Processing account 60 / 515088
Processing account 70 / 515088
Processing account 80 / 515088
Processing account 90 / 515088
Processing account 100 / 515088
Processing account 110 / 515088
Processing account 120 / 515088
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/opt/anaconda3/envs/geo_env/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/2q/6vjsrj112rb774fzn79lb71h0000gn/T/ipykernel_91423/2608950392.py", line 10, in <module>
    x = max(x, (graph.fast_get_money_cycled(account)))
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/graph/builder.py", line 101, in fast_get_money_cycled
    dfs(successor, account, self.LOOP_CUTOFF, 0, simple_paths, [])
  File "/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/graph/builder.py", line 94, in dfs
    dfs(v, target, path_length_threshold - 1, data['timestamp'], paths, current_path)
  File "/Users/zeyaddaowd/Desktop/GP/LedgerDB/aml_service/graph/builder.py", line 94, in dfs
    dfs(v, target, path_length_threshold - 1, data['timestamp'], paths, current_path)
  File "/Users/zeyaddaowd/Desktop